# Grids, Fields, and Planes

Every physicaloptix simulation moves a **field** across **planes**, sampled on
**grids**. These three types are what every propagator and element consumes and
returns, so it is worth building real intuition for them before assembling a
coronagraph.

The physics that ties the planes together is a single Fourier relationship: the
focal-plane field is the Fourier transform of the pupil-plane field,

$$E(\mathbf{x}) = \mathcal{F}\!\left[A(\mathbf{u})\,e^{i\phi(\mathbf{u})}\right],$$

where $A$ is the pupil amplitude (the aperture) and $\phi$ is the wavefront
phase. Everything below is a way of representing the pieces of that equation:
$\mathbf{u}$ and $\mathbf{x}$ are grids, $A e^{i\phi}$ is a field, and the plane
each lives in is a tag the library checks for you.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm, hsv_to_rgb

from physicaloptix import (
    Field,
    Fraunhofer,
    Grid,
    OpticalPath,
    PlaneKind,
    Spectrum,
    Stage,
    broadcast_to_spectrum,
)

WL, DIAM = 600.0, 6.0  # wavelength (nm), aperture diameter (m)


def domain_coloring(field2d, mask):
    """Show a complex field as one image: hue = phase, brightness = amplitude."""
    amp = np.abs(field2d)
    amp = amp / (amp.max() or 1.0)
    hue = (np.angle(field2d) + np.pi) / (2 * np.pi)
    rgb = hsv_to_rgb(np.stack([hue, np.ones_like(hue), amp], axis=-1))
    rgb[~mask] = 1.0  # white outside the aperture
    return rgb

## 1. Two planes, two grids

A `Grid` is an all-static sampling of a plane, defined by just a pixel count
`npix` and a pitch `dx`. The sample coordinates are half-pixel-centered,

$$u_j = \left(j - \tfrac{N}{2} + \tfrac{1}{2}\right)\Delta u, \qquad j = 0,\dots,N-1,$$

so there is deliberately **no sample at the origin** -- the grid straddles the
center. `Grid.pupil(n)` spans one aperture diameter (pupil coordinates run over
roughly $[-\tfrac12, \tfrac12]$); `Grid.focal(n, dx)` is sampled in units of
$\lambda/D$, the natural angular scale of diffraction. The sample points of a
coarse pupil and focal grid look like this:

In [ ]:
pupil = Grid.pupil(16)  # coarse on purpose so the samples are visible
focal = Grid.focal(16, 0.5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, grid, unit, title in (
    (axes[0], pupil, "aperture diameters", "pupil grid"),
    (axes[1], focal, r"$\lambda/D$", "focal grid"),
):
    c = np.asarray(grid.coords)
    gx, gy = np.meshgrid(c, c)
    ax.scatter(gx, gy, s=8, color="C0")
    ax.set_title(f"{title}: npix={grid.npix}, dx={grid.dx:g}")
    ax.set_xlabel(unit)
    ax.set_aspect("equal")
    ax.axhline(0, color="0.7", lw=0.6), ax.axvline(0, color="0.7", lw=0.6)

# zoom on the center to show the half-pixel offset (no sample at the origin)
c = np.asarray(pupil.coords)
gx, gy = np.meshgrid(c, c)
axes[2].scatter(gx, gy, s=40, color="C0")
axes[2].axhline(0, color="C3", lw=1), axes[2].axvline(0, color="C3", lw=1)
axes[2].set_xlim(-2 * pupil.dx, 2 * pupil.dx)
axes[2].set_ylim(-2 * pupil.dx, 2 * pupil.dx)
axes[2].set_title("center zoom: no sample at the origin")
axes[2].set_aspect("equal")
plt.show()

## 2. The Airy pattern sets the natural scale

Propagate a clear circular aperture to the focal plane and you get the Airy
pattern. In the $\lambda/D$ units the focal grid uses, its structure is fixed:
the first dark ring falls at $1.22\,\lambda/D$, the next at $2.23$, then $3.24$.
The intensity is

$$I(r) = \left[\frac{2 J_1(\pi r)}{\pi r}\right]^2, \qquad r \text{ in } \lambda/D.$$

This is why the focal grid is measured in $\lambda/D$: it makes the diffraction
pattern wavelength-independent. The radial cut below lands its nulls exactly on
the theoretical ring radii.

In [ ]:
pupil = Grid.pupil(128)
focal = Grid.focal(320, 0.05)
u = np.asarray(pupil.coords)
ug, vg = np.meshgrid(u, u)
aperture = (np.hypot(ug, vg) <= 0.5).astype(complex)
flat = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.PUPIL)

tele = OpticalPath(stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),))
airy, _ = tele.propagate(flat)
psf = np.abs(np.asarray(airy.data)) ** 2
psf /= psf.max()
fc = np.asarray(focal.coords)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
e = focal.extent
axes[0].imshow(
    psf,
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-6, 1),
    extent=[-e, e, -e, e],
    interpolation="none",
)
axes[0].set_title("Airy pattern"), axes[0].set_xlabel(r"$\lambda/D$")
axes[0].set_xlim(-5, 5), axes[0].set_ylim(-5, 5)

axes[1].semilogy(fc, psf[focal.npix // 2, :], color="C0")
for ring in (1.22, 2.23, 3.24):
    axes[1].axvline(ring, color="C3", ls=":", lw=1)
    axes[1].axvline(-ring, color="C3", ls=":", lw=1)
axes[1].set_xlim(-5, 5), axes[1].set_ylim(1e-6, 1.5)
axes[1].set_title("radial cut: nulls at 1.22, 2.23, 3.24 lambda/D")
axes[1].set_xlabel(r"$\lambda/D$"), axes[1].set_ylabel("normalized intensity")
plt.show()

## 3. A field is complex

A `Field` bundles a complex `data` array with the `grid` it lives on and the
`plane` it occupies. The complex data is where the physics lives:

$$E(\mathbf{u}) = A(\mathbf{u})\,e^{i\phi(\mathbf{u})},$$

with **amplitude** $A = |E|$ (how much light passes -- an aperture, an apodizer)
and **phase** $\phi = \arg E$ (the wavefront shape -- a tilt, a defocus, a mirror
error). One image can show both at once by mapping phase to hue and amplitude to
brightness, shown next to separate amplitude and phase panels.

In [ ]:
u = np.asarray(pupil.coords)
ug, vg = np.meshgrid(u, u)
ap = (np.hypot(ug, vg) <= 0.5).astype(float)
mask = ap > 0
# a tilt plus some defocus, in waves of optical path
opd = 3.0 * ug + 2.0 * (2 * (ug**2 + vg**2) - 0.5)
E = ap * np.exp(1j * 2 * np.pi * opd)

amp = np.where(mask, np.abs(E), np.nan)
phase = np.where(mask, np.angle(E), np.nan)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.0))
axes[0].imshow(domain_coloring(E, mask), origin="lower")
axes[0].set_title("complex field (hue = phase, value = |E|)")
im1 = axes[1].imshow(amp, origin="lower", cmap="gray")
axes[1].set_title("amplitude |E|"), fig.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(phase, origin="lower", cmap="twilight")
axes[2].set_title(r"phase $\arg E$ [rad]"), fig.colorbar(im2, ax=axes[2])
for ax in axes:
    ax.set_xticks([]), ax.set_yticks([])
plt.show()

## 4. What amplitude and phase do downstream

Phase in the pupil becomes structure in the focal plane. A **tilt**
$\phi = 2\pi\,\alpha\,u$ shifts the whole PSF to $\alpha\,\lambda/D$ (this is how
an off-axis planet lands away from the star); **defocus** spreads the core;
**astigmatism** stretches it. Each pupil phase below is shown above the PSF it
produces, so you can read the pupil-to-focal map directly.

In [ ]:
psf_focal = Grid.focal(160, 0.15)
psf_path = OpticalPath(
    stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=psf_focal)),)
)
e = psf_focal.extent

phases = {
    "flat": np.zeros_like(ug),
    "tilt (3 lambda/D)": 3.0 * ug,
    "defocus": 1.2 * (2 * (ug**2 + vg**2) - 0.5),
    "astigmatism": 1.2 * (ug**2 - vg**2),
}
fig, axes = plt.subplots(2, 4, figsize=(13, 6.4))
for col, (name, ph) in enumerate(phases.items()):
    fld = Field(
        data=jnp.asarray(ap * np.exp(1j * 2 * np.pi * ph)),
        grid=pupil,
        plane=PlaneKind.PUPIL,
    )
    out, _ = psf_path.propagate(fld)
    img = np.abs(np.asarray(out.data)) ** 2
    axes[0, col].imshow(np.where(mask, ph, np.nan), origin="lower", cmap="RdBu_r")
    axes[0, col].set_title(name, fontsize=10)
    axes[1, col].imshow(
        img / img.max(),
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-4, 1),
        extent=[-e, e, -e, e],
        interpolation="none",
    )
    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([]), ax.set_yticks([])
axes[0, 0].set_ylabel("pupil phase"), axes[1, 0].set_ylabel("focal PSF")
plt.show()

## 5. The plane tag is a guardrail

The `plane` tag ($\texttt{PUPIL}$, $\texttt{FOCAL}$, $\texttt{INTERMEDIATE}$,
$\texttt{DETECTOR}$) is not decoration. Every propagator declares which plane it
consumes and produces, and it checks the tag when a path is built. A `Fraunhofer`
expects a pupil field; hand it a focal one and it refuses, loudly, instead of
returning a quietly wrong array.

In [ ]:
mis_tagged = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.FOCAL)
try:
    tele.propagate(mis_tagged)
except ValueError as err:
    print("caught, as intended:")
    print(" ", err)

Because a `Field` is an Equinox PyTree, you never mutate it in place. Build a
new one with the constructor, or edit a single leaf with `equinox.tree_at`,
which keeps the grid and plane tag while swapping the data.

## 6. Intensity and energy

A detector sees intensity, $I = |E|^2$; the total power on a plane is the
weighted sum

$$\mathcal{E} = \sum_{ij} |E_{ij}|^2 \, \Delta^2.$$

The continuous-Fourier transform is energy-consistent, so on a well-sampled focal grid the power in the pupil and in the focal plane match -- up to the light that falls outside a finite
field of view. `Field.intensity()` and `Field.energy()` compute these. Below,
the encircled-energy curve shows the fraction of the PSF captured within a given
radius, approaching the pupil energy as the aperture of integration grows.

In [ ]:
wide = Grid.focal(400, 0.1)
wide_path = OpticalPath(
    stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=wide)),)
)
out, _ = wide_path.propagate(flat)
I = np.asarray(out.intensity())

fc = np.asarray(wide.coords)
R = np.hypot(*np.meshgrid(fc, fc))
radii = np.linspace(0, wide.extent, 120)
encircled = np.array([I[R <= r].sum() for r in radii]) * wide.weights

print(f"pupil energy = {float(flat.energy()):.4f}")
print(
    f"focal energy = {float(out.energy()):.4f}  "
    f"({100 * float(out.energy()) / float(flat.energy()):.1f}% of pupil; "
    "the rest is Airy wings beyond the field of view)"
)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(radii, encircled / float(flat.energy()), color="C0")
ax.axhline(1.0, color="0.6", ls=":", lw=1, label="pupil energy")
ax.set_xlabel(r"integration radius [$\lambda/D$]")
ax.set_ylabel("encircled energy / pupil energy")
ax.set_title("energy is conserved into the captured field of view")
ax.legend()
plt.show()

## 7. Chromatic fields

So far every field has been monochromatic. A `Spectrum` attaches a set of
wavelengths and incoherent-sum weights, and `broadcast_to_spectrum` tiles a
field across them into a chromatic `Field` of shape `(nlam, y, x)`. Because the
diffraction scale is $\lambda/D$, the PSF **grows with wavelength**: on a fixed
angular grid, redder light spreads wider. The detector sees the incoherent sum

$$I(\mathbf{x}) = \sum_k w_k \, |E_k(\mathbf{x})|^2,$$

a broadened polychromatic PSF. The overlaid radial cuts make the scaling clear.

In [ ]:
spectrum = Spectrum.tophat(WL, 0.3, 5)  # 5 wavelengths across a 30% band
chromatic = broadcast_to_spectrum(flat, spectrum)
cfocal = Grid.focal(320, 0.05)
cpath = OpticalPath(
    stages=(
        Stage(
            "sci",
            Fraunhofer(grid_in=pupil, grid_out=cfocal, reference_wavelength_nm=WL),
        ),
    )
)
cout, _ = cpath.propagate(chromatic)
per_lambda = np.abs(np.asarray(cout.data)) ** 2
lams = np.asarray(spectrum.wavelengths_nm)
fc = np.asarray(cfocal.coords)
row = cfocal.npix // 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
colors = plt.cm.turbo(np.linspace(0.1, 0.9, len(lams)))
for i, lam in enumerate(lams):
    cut = per_lambda[i, row, :]
    axes[0].plot(fc, cut / cut.max(), color=colors[i], label=f"{lam:.0f} nm")
axes[0].set_xlim(0, 4), axes[0].set_ylim(0, 1.05)
axes[0].set_title("per-wavelength PSF core (redder = wider)")
axes[0].set_xlabel(r"focal position [$\lambda_0/D$]"), axes[0].legend(fontsize=8)

broad = np.asarray(cout.intensity())
e = cfocal.extent
im = axes[1].imshow(
    broad / broad.max(),
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-5, 1),
    extent=[-e, e, -e, e],
    interpolation="none",
)
axes[1].set_xlim(-6, 6), axes[1].set_ylim(-6, 6)
axes[1].set_title("incoherent sum: broadband PSF")
axes[1].set_xlabel(r"$\lambda_0/D$"), fig.colorbar(im, ax=axes[1], label="contrast")
plt.show()

## Where to go next

- **[Propagators and sampling](02_Propagators_and_Sampling)** goes deeper on the
  transform that took each pupil above to its focal plane: how the sampling is
  chosen, how the library checks it, and how a gradient flows through it.
- **[Building a coronagraph](03_Building_a_Coronagraph)** folds masks and stops
  into a path and wraps it as a reusable coronagraph.
- The complex-field and pupil-to-focal intuition from this notebook is the
  foundation for [speckles from first principles](05_Speckles_from_First_Principles),
  where wavefront error scatters starlight into the focal plane.